# Aurora Supply: data → forecast → MLflow
Synthetic CC0 data from 2025. The model learns trend and seasonality. The final 28 days form a chronological holdout. A candidate can lose to the baseline and is then rejected without changing measured metrics.


In [ ]:
from pathlib import Path
import sys, json
repo = Path.cwd()
if repo.name == "notebooks": repo = repo.parent
sys.path.insert(0, str(repo / "scripts"))
from science import generate, train_sku, bundle, PRODUCTS
rows = generate(repo / "data")
print(len(rows), "synthetic rows")


In [ ]:
models = [train_sku(rows, p["sku"]) for p in PRODUCTS]
[(m["sku"], round(m["candidate_mae"], 3), round(m["baseline_mae"], 3), m["model_type"]) for m in models]


In [ ]:
result = bundle(models, rows)
print(json.dumps(result["forecasts"][0], indent=2, ensure_ascii=False))


In [ ]:
%pip install "mlflow[kubernetes]==3.14.0" "boto3==1.43.18"


In [ ]:
from science import publish
# Writes a real experiment and S3 artifact; fails if an integration is unavailable.
published = publish(result)
print(published["mlflow_run_id"])
